# Figure 5 (corrected): bottom slope, bottom vertical velocity and its spectrum, CROCO vs moorings

Replots `plot_figure5.py` with the same data files and conventions as `plot_figure4_corrected.ipynb` / `plot_figureC1_spectra.ipynb` (same moorings, QC, SRTM15 slope `OBS_SLOPE_KM`, model slope `MODEL_SLOPE_KM`, extraction file `FILE_HAB`).

* **a)** bottom slope $|\nabla h|$ of the model; **b)** standard deviation in time of the near-bottom vertical velocity at the lowest model level (map from `extract_bottom_w_std_map.py`); **c)** that standard deviation as a function of the slope (median, 10th and 90th percentiles over the grid) with the moorings (deepest current meter) as markers; **d)** frequency spectrum of one mooring (`PSD_MOORING`) and of CROCO sampled at the same height above bottom.

| `plot_figure5.py` | here |
|---|---|
| slope of the "no smooth" grid computed from `data.h` (the smooth grid, `data_ns.h` is not used) | slope computed from the `h` stored in the std file, with one definition (`SLOPE_MAP_KM`) for the maps, the diagram and the markers |
| the mooring marker uses the model slope at hand-typed grid indices `idx_lon, idx_lat` | indices `i_model, j_model` written by the extraction (nearest model point of the mooring position) |
| mooring std taken as the last entry (`[-1]`) of a precomputed array padded to `nr` bins | `nan`-aware std of the deepest bin (largest median depth), each mooring masked with its own QC flags |
| std of the **true w** of the model compared with the std of $-\mathbf{u}\cdot\nabla D$ of the mooring | `STD_VAR = 'wu'`: std of the same quantity, $-\mathbf{u}\cdot\nabla h$, on both sides (`ALSO_W`: the true-w median curve is added for reference) |
| spectrum: a mooring with a NaN is skipped, a day of 89 400 s, model assumed hourly | see `plot_figureC1_spectra.ipynb` (gappy Welch windows, true cycles per day, model time step read from the file) |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, NullFormatter
from netCDF4 import Dataset
from pathlib import Path

# ---------------- configuration ----------------
DATA_DIR  = Path('./data')                      # folder with the three kinds of files
FILE_D    = DATA_DIR / 'rrex_dD_10km.nc'        # only used as a fallback if the new extraction file has no SRTM15 slopes
OBS_SLOPE_KM   = 10.                            # smoothing length [km] of the OBSERVED slope (SRTM15), among slope_length_km of FILE_HAB
MODEL_SLOPE_KM = 10.                            # smoothing length [km] of the MODEL slope (0 = grid scale), chosen independently of the obs
FILE_CROCO= DATA_DIR / 'rrex_bottom_w_mooring_notracer.nc'   # old extraction (hand-typed hab list)
FILE_HAB  = DATA_DIR / 'rrex_w_moorings_hab_1h.nc'           # extraction used for the first figure: hourly run (use 'rrex_w_moorings_hab.nc' for the 12-hourly run)
MOORINGS  = ['IRW', 'IRM', 'IRE', 'RRT', 'ICW', 'ICM', 'ICE']    # index in dDdx, wb*, wub* = position in this list

# model quantities to plot (the first is the main one; the others are drawn on top in other colours).  New file:
#   'wu_L' : -u.grad(H smoothed over MODEL_SLOPE_KM)     'wu' : -u.grad(H), grid-scale slope     'w' : true model vertical velocity
# Old file (no new extraction): 'wu_L'/'wu' -> 'wub', 'w' -> 'wb'.
MODEL_SERIES = ['wu_L', 'w']                                  # keep both: figure 1 uses 'wu_L' (or 'w' if MOORING_VAR='WCUR'), figure 2 uses 'wu_L' and 'w'
MOORING_VAR = 'wu'      # 'wu' : -u.grad(D) from the deepest current meter ; 'WCUR': measured vertical velocity
QC_MAX      = 2         # keep samples with QC flag <= QC_MAX (1 = good, 2 = probably good)

# surveyed water depth at the mooring [m] (depth_cv of the old extraction script): reference for the observed hab
DEPTH_REF   = np.array([2160., 1570., 1580., 1528., 2162., 2151., 2379.])
# hab list typed by hand in the OLD extraction (what the old file was sampled at)
HAB_EXTRACT = np.array([100, 93, 116, 81, 57, 46, 282])

LOG_Y = True            # True: logarithmic pdf axis (shows the tails); False: linear axis (figures saved without the '_log' suffix)
YMIN, YMAX = 5e-5, 0.1   # y limits of the log axis, in probability density [(m/day)^-1]
YLIM_LIN = None          # y limits of the linear axis, e.g. (0, 0.03); None = automatic

dw  = 2.                                            # bin width [m/day]
w_e = np.arange(-750, 750 + dw, dw)
XLIM = (-310, 310)
SEC_PER_DAY = 86400.
fs = 14

# ---------------- spectra ----------------
SPEC_SERIES   = ['wu_L']      # model series drawn: 'wu_L' (-u.grad H smoothed over MODEL_SLOPE_KM), 'wu' (grid scale) and/or 'w' (true model w)
MODEL_DT_H    = None          # time step of the model snapshots [hours]: None = read from the extraction file (attribute dt_s; 12 h if absent)
OBS_DT_H      = 1.0           # time step of the mooring files [hours]
NPERSEG_DAYS  = 30.           # Welch window length [days] (shortened if the record is shorter)
SEASON        = (10, 11)          # None: whole mooring record.  (9, 3): only the months September to March (both included); used for the MOORING SPECTRUM of panel d) only (panel c) uses the whole record)
SHOW_CI       = True          # panel d): shaded confidence interval of the spectra
CI_LEVEL      = 0.95          # confidence level
MAX_NAN_FRAC  = 0.02          # a window is kept if at most this fraction of its samples is NaN (they are linearly interpolated)
OMEGA         = 7.2921e-5     # Earth rotation rate [rad s-1]
YLIM_PSD      = (1e-7, 10)
FS_PLOT       = 20
MONTH_NAMES   = 'Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec'.split()
SEASON_TAG    = '' if SEASON is None else '_%s%s' % (MONTH_NAMES[SEASON[0] - 1], MONTH_NAMES[SEASON[1] - 1])    # suffix of the saved figure
MOOR_LABEL    = 'Mooring' + ('' if SEASON is None else ' (%s-%s)' % (MONTH_NAMES[SEASON[0] - 1], MONTH_NAMES[SEASON[1] - 1]))
MODEL_SERIES = list(SPEC_SERIES)    # series loaded from the extraction file

# ---------------- second figure: 12-hourly and hourly model output on the same panels ----------------
FILE_HAB_12H  = DATA_DIR / 'rrex_w_moorings_hab.nc'          # extraction from rrexnumsb200_RSUP5_NOFILT_tracer (12 h output)
FILE_HAB_1H   = DATA_DIR / 'rrex_w_moorings_hab_1h.nc'       # extraction from ..._tracer_article_hourly (1 h output)
COMPARE_SERIES = 'wu_L'                                      # model quantity compared: 'wu_L' (MODEL_SLOPE_KM), 'wu' (grid scale) or 'w'


# ---------------- Figure 5 ----------------
FILE_STD     = DATA_DIR / 'rrex_bottom_w_std_map.nc'   # extract_bottom_w_std_map.py (12 h run); '..._1h.nc' for the hourly run
STD_HAB      = None            # None: lowest model level; 50. or 100.: height above bottom [m] (needs HAB_EXTRA of extract_bottom_w_std_map.py)
STD_VAR      = 'w'             # 'w': std of the true model w (panels b, c)   'wu': std of -u.grad(h) (same quantity as the mooring)
ALSO_W       = True            # panel c: also draw the median std of the true w (dashed)
SLOPE_MAP_KM = 0.              # smoothing length [km] of h for the model slope of the maps and the diagram (0: grid scale)
MARKER_SLOPE = 'model'         # slope of the mooring markers in c): 'model' (same as the curves) or 'obs' (SRTM15, OBS_SLOPE_KM)
PSD_MOORING  = 'RRT'           # mooring of panel d)
SLOPE_MAX    = 0.12            # slope axis / colour range
STD_MAX      = 200.            # std colour range [m/day]
SLOPE_BINS   = np.arange(0., SLOPE_MAX + 1e-9, 0.002)


In [ ]:
def masked_to_nan(a):
    """netCDF masked array -> float array with NaN for masked / fill values."""
    a = np.ma.filled(np.ma.asarray(a).astype(float), np.nan)
    a[np.abs(a) > 1e5] = np.nan                  # 999999 fill values
    return a

def read_mooring(name, i_grad, grad):
    ds = Dataset(DATA_DIR / ('RREX_%s_CURR_hourly.nc' % name))
    depth = masked_to_nan(ds['DEPTH'][:])
    zmed  = np.nanmedian(depth, axis=0)
    k     = int(np.nanargmax(zmed))                   # deepest bin
    out   = dict(name=name, k=k, nlev=depth.shape[1], k_is_last=(k == depth.shape[1] - 1), z=zmed[k])
    def get(var):
        x = masked_to_nan(ds[var][:, k])
        if var + '_QC' in ds.variables:
            q = np.ma.filled(ds[var + '_QC'][:, k], 9)
            x[q > QC_MAX] = np.nan
        return x
    u, v = get('UCUR'), get('VCUR')
    out['u'], out['v'] = u, v                                          # [m s-1] for the horizontal-velocity spectra
    out['wu']   = -(u * grad['dDdx'][i_grad] + v * grad['dDdy'][i_grad]) * SEC_PER_DAY   # D > 0: bottom at z = -D
    out['WCUR'] = get('WCUR') * SEC_PER_DAY
    out['lat'] = float(np.ravel(masked_to_nan(ds['LATITUDE'][:]))[0])
    # optional season: samples outside the months SEASON = (first, last) (both included, may wrap over the new year) -> NaN
    days = np.ma.filled(ds['TIME'][:], np.nan).astype(float)                        # days since 1950-01-01
    month = (np.datetime64('1950-01-01') + (days * 86400).astype('timedelta64[s]')).astype('datetime64[M]').astype(int) % 12 + 1
    out['time'], out['month'] = days, month
    keep = np.ones(len(month), bool)                                               # samples used by the SPECTRA (panel d); panel c uses the whole record
    if SEASON is not None:
        m1, m2 = SEASON
        keep = (month >= m1) & (month <= m2) if m1 <= m2 else (month >= m1) | (month <= m2)
    out['keep_season'] = keep
    out['season_frac'] = float(keep.mean())
    ds.close()
    return out

# observed bottom slope at the moorings: SRTM15 slope smoothed over OBS_SLOPE_KM, computed by the extraction script
if FILE_HAB.exists() and 'dDdx_topo15' in Dataset(FILE_HAB).variables:
    with Dataset(FILE_HAB) as nc:
        L = np.asarray(nc['slope_length_km'][:], float)
        k = int(np.argmin(abs(L - OBS_SLOPE_KM)))
        grad = {'dDdx': np.asarray(nc['dDdx_topo15'][k], float), 'dDdy': np.asarray(nc['dDdy_topo15'][k], float),
                'D': np.asarray(nc['D_topo15'][:], float)}
    print('observed slope: SRTM15 smoothed over %g km (from %s)' % (L[k], FILE_HAB.name))
else:
    with Dataset(FILE_D) as nd:                                 # fallback: precomputed slopes (method not documented)
        grad = {k: np.asarray(nd[k][:], float) for k in ('D', 'dDdx', 'dDdy')}
    print('observed slope: %s (no SRTM15 slopes found)' % FILE_D.name)
obs = [read_mooring(m, i, grad) for i, m in enumerate(MOORINGS)]

USE_NEW = FILE_HAB.exists()
mod, mod_label = {}, {}                                          # mod[name] = list over moorings of the model series
if USE_NEW:
    with Dataset(FILE_HAB) as nc:
        Lm = np.asarray(nc['slope_length_km'][:], float) if 'slope_length_km' in nc.variables else np.array([])
        for name in MODEL_SERIES:
            if name == 'w':
                arr, mod_label[name] = nc['w_obs_hab'][:], r'CROCO: $w$'
            elif name == 'wu' or (name == 'wu_L' and MODEL_SLOPE_KM == 0 and 'wu_L_obs_hab' not in nc.variables):
                arr, mod_label[name] = nc['wu_obs_hab'][:], r'CROCO: $-\mathbf{u}\cdot\nabla H$ (grid scale)'
            elif 'wu_L_obs_hab' in nc.variables:
                kL = int(np.argmin(abs(Lm - MODEL_SLOPE_KM)))
                assert abs(Lm[kL] - MODEL_SLOPE_KM) < 1e-6, 'MODEL_SLOPE_KM=%g not in %s' % (MODEL_SLOPE_KM, Lm)
                arr, mod_label[name] = nc['wu_L_obs_hab'][:, :, kL], r'CROCO: $-\mathbf{u}\cdot\nabla H$'
            else:                                                # file from the previous version of the extraction
                print('wu_L_obs_hab not in %s: using wu_s (smoothing %g km)' % (FILE_HAB.name, nc.smooth_km))
                arr, mod_label[name] = nc['wu_s_obs_hab'][:], r'CROCO: $-\mathbf{u}\cdot\nabla H$'
            arr = np.asarray(np.ma.filled(np.ma.asarray(arr).astype(float), np.nan), float)    # (time, mooring)
            mod[name] = [arr[:, i] for i in range(len(MOORINGS))]
        hab_model = np.asarray(nc['hab_obs'][:], float)           # hab at which the model was sampled
        h_model   = np.asarray(nc['h'][:], float)
    print('model file: %s (sampled at hab_obs), series %s' % (FILE_HAB.name, ', '.join(MODEL_SERIES)))
else:
    with Dataset(FILE_CROCO) as nc:
        for name in MODEL_SERIES:
            old = 'wb' if name == 'w' else 'wub'
            mod[name] = [np.asarray(nc['%s%d' % (old, i)][:], float) for i in range(len(MOORINGS))]
            mod_label[name] = r'CROCO: $w$' if name == 'w' else r'CROCO: $-\mathbf{u}\cdot\nabla H$'
    hab_model = HAB_EXTRACT.astype(float)
    h_model   = None
    print('%s not found: using the OLD file %s, sampled at the hand-typed hab list' % (FILE_HAB.name, FILE_CROCO.name))


In [ ]:
def clean(x): return x[np.isfinite(x)]
print('%-4s %8s %10s | %8s | %10s %9s' % ('', 'z_sensor', 'depth_ref', 'hab_obs', 'hab model', 'model-obs'))
for i, o in enumerate(obs):
    o['hab'] = DEPTH_REF[i] - o['z']                  # height of the deepest bin above the surveyed bottom
    d = hab_model[i] - o['hab']
    print('%-4s %8.1f %10.0f | %8.1f | %10.1f %9.1f' % (o['name'], o['z'], DEPTH_REF[i], o['hab'], hab_model[i], d))
print()
print('hab_obs   = depth_ref - z_sensor : height of the deepest current meter above the surveyed bottom')
print('hab model = height above the MODEL floor at which the model series was sampled')
bad = [o['name'] for i, o in enumerate(obs) if abs(hab_model[i] - o['hab']) > 5]
if bad:
    print('\n!! model and mooring are NOT at the same hab (> 5 m apart) for: %s' % ', '.join(bad))
    print('   -> run extract_bottom_w_moorings_croco_hab.py, which samples the model at hab_obs')
else:
    print('\nmodel and mooring are at the same height above bottom (within 5 m) at all moorings')

In [ ]:
import scipy.signal as sig
from scipy.stats import chi2


class Spec(tuple):
    """(frequency, psd) that also carries `nu`, the equivalent degrees of freedom of the estimate (for the confidence interval)."""
    def __new__(cls, fp, nu):
        o = super().__new__(cls, fp); o.nu = nu; return o


def edof(K, mult=1):
    """Equivalent degrees of freedom of a Welch estimate averaging K segments with 50 % overlap and a Hann window (Welch 1967):
    nu = 36 K^2 / (19 K - 1)  (~1.9 K for large K)."""
    return mult * 36. * K ** 2 / (19. * K - 1.) if K > 0 else np.nan


def ci_band(ax, sp, color):
    """Shaded CI_LEVEL confidence interval of the spectrum `sp` (chi-square law with sp.nu degrees of freedom, multiplicative on a log axis)."""
    nu = getattr(sp, 'nu', np.nan)
    if not SHOW_CI or not np.isfinite(nu):
        return
    a = 1. - CI_LEVEL
    lo, hi = nu / chi2.ppf(1. - a / 2., nu), nu / chi2.ppf(a / 2., nu)
    f, p = sp
    ax.fill_between(cpd(f[1:]), p[1:] * lo, p[1:] * hi, color=color, alpha=0.2, lw=0)


def welch_gappy(x, dt_s, nperseg_days):
    """Welch spectrum [x^2 Hz^-1] of a series with gaps: non-overlapping windows with at most MAX_NAN_FRAC NaN are
    interpolated and averaged (Hann window, mean removed).  The window is shortened to a quarter of the record if the record is
    shorter than nperseg_days.  Returns (freq [Hz], psd, number of windows, window length [days])."""
    x = np.asarray(x, float)
    n = int(round(nperseg_days * 86400. / dt_s))
    if len(x) < 2 * n:                                   # record too short: use windows of a quarter of the record
        n = max(len(x) // 4, 8)
    specs = []
    for i0 in range(0, len(x) - n + 1, n // 2):          # 50 % overlap
        w = x[i0:i0 + n].copy()
        bad = ~np.isfinite(w)
        if bad.mean() > MAX_NAN_FRAC or bad.all():
            continue
        if bad.any():
            w[bad] = np.interp(np.flatnonzero(bad), np.flatnonzero(~bad), w[~bad])
        f, p = sig.welch(w, 1. / dt_s, window='hann', nperseg=n, detrend='constant')
        if n % 2 == 0:                                   # scipy does not double the Nyquist bin of a one-sided spectrum -> correct it
            p = p.copy(); p[-1] *= 2.
        specs.append(p)
    if not specs:
        return np.array([np.nan]), np.array([np.nan]), 0, n * dt_s / 86400.
    return f, np.mean(specs, axis=0), len(specs), n * dt_s / 86400.

dt_file = None
if USE_NEW:
    with Dataset(FILE_HAB) as nc:
        dt_file = getattr(nc, 'dt_s', None)
dt_mod = MODEL_DT_H * 3600. if MODEL_DT_H else (float(dt_file) if dt_file else 12 * 3600.)
dt_obs = OBS_DT_H * 3600.
print('model time step: %g h (%s) -> Nyquist %.2f cpd' % (dt_mod / 3600., 'file' if (dt_file and not MODEL_DT_H) else 'default/config', 43200. / dt_mod))
spec_obs, spec_mod = [], {n: [] for n in SPEC_SERIES}
print('%-4s %-8s %7s %8s %9s' % ('', 'series', 'samples', 'windows', 'window[d]'))
for i, o in enumerate(obs):
    x = np.where(o['keep_season'], o[MOORING_VAR], np.nan) / SEC_PER_DAY      # m/day -> m/s; only the months of SEASON (spectra only)
    f, p, nw, wd = welch_gappy(x, dt_obs, NPERSEG_DAYS)
    spec_obs.append(Spec((f, p), edof(nw))); print('%-4s %-8s %7d %8d %9.1f' % (o['name'], 'mooring', len(x), nw, wd))
    for name in SPEC_SERIES:
        xm = mod[name][i] / SEC_PER_DAY
        f, p, nw, wd = welch_gappy(xm, dt_mod, NPERSEG_DAYS)
        spec_mod[name].append(Spec((f, p), edof(nw))); print('%-4s %-8s %7d %8d %9.1f' % ('', name, len(xm), nw, wd))


In [ ]:
from scipy.ndimage import uniform_filter
import scipy.stats as stats

with Dataset(FILE_STD) as nc:
    suf = '' if STD_HAB is None else '_hab%g' % STD_HAB
    g = {k: np.ma.filled(np.ma.asarray(nc[k + suf if k.startswith('std') else k][:]).astype(float), np.nan) for k in ('std_w', 'std_wu', 'h', 'pm', 'pn', 'mask')}
    std_sim = getattr(nc, 'simulation', '?')
nx, ny = g['h'].shape
ocean = (g['mask'] > 0) & np.isfinite(g['std_w'])
h_m = np.where(g['mask'] > 0, g['h'], np.nan)

def model_slope(h, pm, pn, length_km):
    """|grad h| of the model (boxcar average over length_km if > 0, then centred differences)."""
    hh = np.where(np.isfinite(h), h, np.nanmean(h))
    if length_km > 0:
        n = (max(1, int(round(length_km * 1e3 * np.nanmean(pm)))), max(1, int(round(length_km * 1e3 * np.nanmean(pn)))))
        hh = uniform_filter(hh, size=n, mode='nearest')
    return np.hypot(np.gradient(hh, axis=0) * pm, np.gradient(hh, axis=1) * pn)

gradh = np.where(ocean, model_slope(g['h'], g['pm'], g['pn'], SLOPE_MAP_KM), np.nan)
std_map = np.where(ocean, g['std_' + STD_VAR], np.nan)
std_w_map = np.where(ocean, g['std_w'], np.nan)
xkm = np.cumsum(1. / g['pm'][:, ny // 2]) / 1e3
ykm = np.cumsum(1. / g['pn'][nx // 2, :]) / 1e3
print('std file: %s (simulation %s), %d x %d points, %d ocean' % (FILE_STD.name, std_sim, nx, ny, ocean.sum()))

# statistics of the std(w) in slope bins
ok = np.isfinite(gradh) & np.isfinite(std_map)
def binned(field, how):
    return stats.binned_statistic(gradh[ok], field[ok], statistic=how, bins=SLOPE_BINS)[0]
slope_c = 0.5 * (SLOPE_BINS[1:] + SLOPE_BINS[:-1])
p10 = lambda y: np.nanpercentile(y, 10); p90 = lambda y: np.nanpercentile(y, 90)
curves = {'median': binned(std_map, np.nanmedian), 'p10': binned(std_map, p10), 'p90': binned(std_map, p90)}
curve_w = stats.binned_statistic(gradh[ok], std_w_map[ok], statistic=np.nanmedian, bins=SLOPE_BINS)[0]

# moorings: std of -u.grad(D) at the deepest bin, and slope
std_obs = np.array([np.nanstd(o['wu']) for o in obs])               # m/day
with Dataset(FILE_HAB) as nc:
    i_mod, j_mod = np.asarray(nc['i_model'][:], int), np.asarray(nc['j_model'][:], int)
if MARKER_SLOPE == 'model':
    slope_m = gradh[i_mod, j_mod]
else:
    slope_m = np.hypot(grad['dDdx'], grad['dDdy'])
print('%-4s %12s %10s %12s' % ('', 'std obs [m/d]', 'slope', 'model std [m/d]'))
for k, o in enumerate(obs):
    print('%-4s %12.1f %10.4f %12.1f' % (o['name'], std_obs[k], slope_m[k], std_map[i_mod[k], j_mod[k]]))


In [ ]:
import matplotlib.colors as colors
fsz = 24                                   # base font size (labels, ticks, titles)
DAY = 86400.
TIDES = {'O$_1$': 1. / 92950., 'K$_1$': 1. / 86164., 'M$_2$': 1. / 44700., 'S$_2$': 1. / 43200.}
PERIODS_D = {'month': 365.25 / 12, 'week': 7.}          # long periods marked on the spectrum [days]
cpd = lambda f_hz: np.asarray(f_hz) * DAY
cf0 = colors.to_rgba('teal') if STD_VAR == 'w' else colors.to_rgba('royalblue')
std_label = r'STD($-\mathbf{u}\cdot\nabla h$) [m day$^{-1}$]' if STD_VAR == 'wu' else r'STD($w$) [m day$^{-1}$]'
mcolors = plt.cm.magma(np.linspace(0.2, 0.85, len(obs)))

# ---- layout in inches: four panels of identical size, with the smallest gaps that keep labels and colourbars readable ----
FIG_W   = 20.
M_L, M_R, M_B, M_T = 1.6, 0.3, 1.1, 0.6      # margins: left (y label + ticks), right, bottom (x label + ticks), top (title)
GAP_X   = 1.9                                 # between columns (y label + ticks of the right column)
GAP_Y   = 3.3                                 # between rows (x label + ticks + colourbar + its label + title of the lower row)
BOX     = (ykm.max() - ykm.min()) / (xkm.max() - xkm.min())        # box aspect of the maps; all panels get it
PW      = (FIG_W - M_L - M_R - GAP_X) / 2.
PH      = BOX * PW
FIG_H   = M_B + 2 * PH + GAP_Y + M_T
fig = plt.figure(figsize=(FIG_W, FIG_H))
def panel(col, row):                           # row 0 = top
    x0 = M_L + col * (PW + GAP_X)
    y0 = M_B + (1 - row) * (PH + GAP_Y)
    return fig.add_axes([x0 / FIG_W, y0 / FIG_H, PW / FIG_W, PH / FIG_H])
def cbar_axes(ax):                             # horizontal colourbar below a map: 1.3 in under the axis, 0.3 in thick
    b = ax.get_position()
    return fig.add_axes([b.x0, b.y0 - 1.6 / FIG_H, b.width, 0.3 / FIG_H])

def bathy(ax, col='k', levels=np.arange(500, 4500, 500)):
    ax.contour(xkm, ykm, h_m.T, levels=levels, colors=col, linewidths=0.4)

# a) slope map
ax = panel(0, 0); ax.set_xlim(xkm.min(), xkm.max()); ax.set_ylim(ykm.min(), ykm.max())
pa = ax.pcolormesh(xkm, ykm, gradh.T, cmap=plt.cm.copper_r, norm=colors.Normalize(0, SLOPE_MAX), shading='auto')
bathy(ax)
for k, o in enumerate(obs):
    ax.plot(xkm[i_mod[k]], ykm[j_mod[k]], 'D', color='w', mec='k', ms=6)
    ax.text(xkm[i_mod[k]] + 8, ykm[j_mod[k]] + 8, o['name'], fontsize=0.6 * fsz, color='k')
ax.set_title('a)', fontsize=fsz, pad=6); ax.set_xlabel(r'km in $\xi$-direction', fontsize=fsz); ax.set_ylabel(r'km in $\eta$-direction', fontsize=fsz)
ax.tick_params(labelsize=fsz)
cb = fig.colorbar(pa, cbar_axes(ax), orientation='horizontal', ticks=[0, SLOPE_MAX / 2, SLOPE_MAX])
cb.set_label('slope' + ('' if SLOPE_MAP_KM == 0 else ' (smoothed over %g km)' % SLOPE_MAP_KM), fontsize=fsz); cb.ax.tick_params(labelsize=fsz)

# b) std map
ax = panel(1, 0); ax.set_xlim(xkm.min(), xkm.max()); ax.set_ylim(ykm.min(), ykm.max())
pb = ax.pcolormesh(xkm, ykm, std_map.T, cmap=plt.cm.nipy_spectral, norm=colors.Normalize(0, STD_MAX), shading='auto')
bathy(ax, 'gray', [1000, 2000, 3000, 4000])
ax.set_title('b)', fontsize=fsz, pad=6); ax.set_xlabel(r'km in $\xi$-direction', fontsize=fsz); ax.set_yticklabels([]); ax.tick_params(labelsize=fsz)
cb = fig.colorbar(pb, cbar_axes(ax), orientation='horizontal', extend='both', ticks=[0, 50, 100, 150, STD_MAX])
cb.set_label(std_label + (' at the lowest model level' if STD_HAB is None else ' at %g m above bottom' % STD_HAB), fontsize=0.9 * fsz); cb.ax.tick_params(labelsize=fsz)

# c) std as a function of the slope
ax = panel(0, 1)
ax.plot(curves['median'], slope_c, color=cf0, lw=2.5, label='Median (CROCO)')
ax.plot(curves['p10'], slope_c, color=cf0, ls='--', lw=2.5, label=r'$10^{th}$, $90^{th}$ percentiles')
ax.plot(curves['p90'], slope_c, color=cf0, ls='--', lw=2.5)
if ALSO_W and STD_VAR != 'w':
    ax.plot(curve_w, slope_c, color='teal', ls=':', lw=2.5, label=r'Median, STD($w$) (CROCO)')
for k, o in enumerate(obs):
    ax.scatter(std_obs[k], slope_m[k], s=90, marker='D', color=mcolors[k], edgecolor='k', zorder=5,
               label='Moorings (deepest measure)' if k == 0 else None)
    ax.annotate(o['name'], (std_obs[k], slope_m[k]), xytext=(6, 4), textcoords='offset points', fontsize=0.7 * fsz, color=mcolors[k])
ax.set_xscale('log'); ax.set_xlim(2, 1000); ax.set_ylim(0, SLOPE_MAX)
ax.set_yticks(np.arange(0, SLOPE_MAX + 1e-9, 0.02)); ax.set_yticklabels([str(int(round(v * 100))) for v in np.arange(0, SLOPE_MAX + 1e-9, 0.02)])
ax.grid(True, which='both', axis='x', alpha=0.25); ax.grid(True, which='major', alpha=0.5)
ax.set_xlabel(std_label, fontsize=fsz); ax.set_ylabel('Slope [%]', fontsize=fsz); ax.set_title('c)', fontsize=fsz, pad=6)
ax.tick_params(labelsize=fsz); ax.legend(fontsize=0.7 * fsz, loc='upper left')

# d) frequency spectrum of one mooring
ax = panel(1, 1)
im = [o['name'] for o in obs].index(PSD_MOORING)
sp = spec_obs[im]; f, p = sp
ax.loglog(cpd(f[1:]), p[1:], color='tab:red', lw=1.5, label=MOOR_LABEL)
ci_band(ax, sp, 'tab:red')
for name in SPEC_SERIES:
    sp = spec_mod[name][im]; f, p = sp
    ax.loglog(cpd(f[1:]), p[1:], color='tab:blue', lw=1.5, alpha=0.75, label=mod_label[name])
    ci_band(ax, sp, 'tab:blue')
fi = 2 * OMEGA * np.sin(np.deg2rad(obs[im]['lat'])) / (2 * np.pi)
for lab, fr in list(TIDES.items()) + [('$f$', fi)]:
    ax.axvline(cpd(fr), color='k', ls='--', lw=0.9)
for lab, per in PERIODS_D.items():                                   # monthly and weekly periods
    ax.axvline(1. / per, color='tab:green', ls=':', lw=1.6)
yl = YLIM_PSD
ytxt = yl[1] * 0.6
ax.text(cpd(fi), ytxt, '$f$', ha='right', va='top', fontsize=0.75 * fsz)
ax.text(0.5 * (cpd(TIDES['O$_1$']) + cpd(TIDES['K$_1$'])), ytxt, 'O$_1$,K$_1$', ha='center', va='top', fontsize=0.75 * fsz)
ax.text(0.5 * (cpd(TIDES['M$_2$']) + cpd(TIDES['S$_2$'])) * 1.02, ytxt, 'M$_2$,S$_2$', ha='left', va='top', fontsize=0.75 * fsz)
for lab, per in PERIODS_D.items():
    ax.text(1. / per * 1.04, ytxt, lab, ha='left', va='top', fontsize=0.75 * fsz, color='tab:green')
ax.set_ylim(*yl); ax.set_xlim(0.025, 14)
ax.grid(True, which='major', alpha=0.5); ax.grid(True, which='minor', alpha=0.2, lw=0.5)
ax.set_xlabel('Cycle per day', fontsize=fsz); ax.set_ylabel(r'PSD [(m s$^{-1}$)$^2$ Hz$^{-1}$]', fontsize=fsz)
ax.set_title('d) %s' % PSD_MOORING, fontsize=fsz, pad=6); ax.tick_params(labelsize=fsz); ax.legend(fontsize=0.7 * fsz, loc='lower left')
fig.savefig('figure5_corrected' + SEASON_TAG + '.png', dpi=180, bbox_inches='tight')
plt.show()
